# 02 · 프롬프트 설계
**블록 3 · 프롬프트 설계** — 실습 60분

여기서 만드는 v1이 오후 블록 4에서 20문항으로 채점받는 **1차**의 재료입니다. 지금은 그 전에 "쓸 만한 프롬프트 하나"를 준비하는 단계입니다.

사내 문서 한 편에서 **「제목 · 항목 3개 · 단위」**를 뽑아내는 프롬프트를 씁니다.

```json
{"제목": "2026년 4분기 사무용품 구매 계획",
 "항목": [{"이름": "복사용지", "값": 120, "단위": "박스"},
          {"이름": "토너", "값": 8, "단위": "개"},
          {"이름": "예산", "값": 340, "단위": "만원"}]}
```

**통과 기준** — 10번 돌려 10번 모두 `json.loads` 통과 · 항목이 정확히 3개 · 값은 숫자, 단위는 문자열 · 문서에 없는 값은 지어내지 않음

| 순서 | 할 일 | 시간 |
|---|---|---|
| 1 | 문서 고르기 — 업무 문서 한 편과 뽑을 항목 세 개 (없으면 `data/docs/`) | 10분 |
| 2 | v0 — 한 줄짜리 지시로 먼저 돌려 보고 무엇이 부족한지 적는다 | 10분 |
| 3 | v1 — 프롬프트 파일을 한 줄씩 채우고, 채울 때마다 돌려 본다 | 25분 |
| 4 | 짝 점검 — 체크리스트로 서로의 v1을 보고, 10번 돌려 형식 확인 | 15분 |

In [ ]:
import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")          # day 폴더로 이동 — data/ · prompts/ · src/ 가 여기 있다

from common import llm      # 공용 API 도구 (ETRI_LLM_3days/common/llm.py)
print("모델:", llm.MODEL, "| 모의 모드" if llm.MOCK else "| 실제 호출")
import json
from src.grade import build, parse, check

## 1. 문서 고르기

In [ ]:
DOC_PATH = "data/docs/01_사무용품_구매계획.txt"   # ✏️ 내 업무 문서로 바꿔도 됩니다
doc = Path(DOC_PATH).read_text(encoding="utf-8")
print(doc)

## 2. v0 — 한 줄짜리 지시
`prompts/prompt_v0.txt`를 그대로 돌립니다. 결과를 보고 **무엇이 부족한지** 적습니다.

In [ ]:
def try_prompt(prompt_text, document, n=1, show=True):
    """프롬프트를 n번 돌려 JSON 파싱 성공 횟수를 센다."""
    system, user = build(prompt_text, document)
    if show:
        print(f"=== system에 들어간 것 ===\n{system}\n")
        print(f"=== user에 들어간 것 ===\n{user}\n")
    ok = 0
    for i in range(n):
        r = llm.call(user, system=system, max_tokens=1000)
        try:
            out = parse(r.text); ok += 1; note = f"JSON OK · 항목 {len(out.get('항목', []))}개"
        except ValueError as e:
            note = f"JSON 실패 — {e}"
        if show:
            print(f"--- {i+1}회 · {note} · 출력 {r.output_tokens} 토큰")
            print(r.text.strip()[:600])
    print(f"\n파싱 성공 {ok} / {n}")
    return ok

v0 = llm.load_prompt("prompts/prompt_v0.txt")
print(v0, "\n========")
try_prompt(v0, doc)

**v0에서 부족했던 것** (한 줄씩):
-
-
- 

## 3. v1 — 아래 표를 한 줄씩 채우고 바로 돌려본다
`prompts/prompt_v1.txt`를 엽니다. 표의 항목을 **위에서부터 한 줄씩** 채우고, 한 줄 채울 때마다 바로 아래 셀을 다시 실행해 뭐가 달라지는지 봅니다. 다 채운 뒤에 한꺼번에 실행하지 않습니다 — 그러면 뭐가 효과 있었는지 알 수 없습니다.

| 채울 내용 | 넣는 곳 |
|---|---|
| 역할을 구체적으로 — 누가, 무엇을 위해 | `<역할>` |
| 할 일을 구체적으로 — 몇 개를, 어떤 기준으로 뽑는지 | `<할 일>` |
| "~하지 마" 대신 "~해라"로 쓰기 | `<역할>` `<할 일>` 전체 |
| 예시 하나 — 짧은 문서와 정답 JSON | `<예시>` |
| 원하는 JSON 모양을 그대로 보여주기 | `<출력 형식>` |

파일 맨 위 `=== system ===` 아래는 system으로, `=== user ===` 아래는 user로 보냅니다. `{document}` 자리에 문서가 들어갑니다.

In [ ]:
v1 = llm.load_prompt("prompts/prompt_v1.txt")      # 파일을 고친 뒤 이 셀을 다시 실행
try_prompt(v1, doc)

> 막히면 `solutions/prompts/prompt_v1.txt`를 열어 봅니다. 그대로 베끼기보다, 내 v1과 **무엇이 다른지** 비교합니다.

## 4. 짝 점검 체크리스트
v1을 짝과 바꿔 보고, 아래 다섯 가지를 하나씩 확인합니다. 하나라도 "아니오"면 3절로 돌아가 고칩니다.

- [ ] **형식** — 10번 돌려 10번 다 `json.loads`로 읽히는가? (코드블록·설명 없이)
- [ ] **개수** — 항목이 항상 정확히 3개인가? (많지도 적지도 않게)
- [ ] **타입** — 값은 숫자, 단위는 문자열인가?
- [ ] **지어냄 금지** — 문서에 없는 수치를 채워 넣지 않는가?
- [ ] **구조** — `=== system ===`에는 역할·규칙·형식만, `=== user ===`에는 문서만 들어가 있는가?

아래 셀은 이 중 **첫 번째(형식)만 자동으로** 확인합니다. 나머지 네 개는 사람이 눈으로 봐야 합니다 — 9번 셀에서 연 원문을 같이 보면서 짝과 확인하세요.

In [ ]:
ok = try_prompt(v1, doc, n=10, show=False)
print("통과" if ok == 10 else "아직 — 형식 지시 · 예시를 다시 봅니다")

## 5. 생각할 자리를 준다
**방금 만든 v1에, 한 가지를 더합니다.** 바로 `<json>`을 쓰게 하지 말고, **근거를 먼저 옮겨 적게** 합니다. 출력 형식은 v1 것을 그대로 쓰고, 규칙만 하나 추가합니다.

코드는 `<json>` 안만 꺼내 쓰고(v1과 같은 `parse()`), `<근거>`는 사람이 디버깅할 때 엽니다.

In [ ]:
import re

system, user = build(v1, doc)      # v1의 <출력 형식>을 그대로 쓴다
EVIDENCE = ("\n\n<근거 규칙>\n<json>을 쓰기 전에 <근거>에 문서에서 실제로 찾은 문장을 그대로 "
            "옮겨 적어라. 문서에 없으면 그 항목은 <json>에 넣지 않는다.\n</근거 규칙>")

r = llm.call(user, system=(system or "") + EVIDENCE, max_tokens=800)
print(r.text.strip())

m = re.search(r"<근거>(.*?)</근거>", r.text, re.S)
print("\n→ 코드가 디버깅용으로 꺼낸 <근거>:", (m.group(1).strip() if m else "(없음)"))
print("→ parse() 결과(<json>만 읽음):", parse(r.text))

## 6. 한 덩어리 일을 세 단계로 나눈다 — 뽑기(모델) → 검사(코드) → 쓰기(모델)
이 절은 프롬프트를 잘 쓰는 기술이 아니라 **작업을 어떻게 나눌지**에 대한 얘기입니다. 프롬프트로 해결하려 하지 말고 코드로 해야 하는 부분이 있다는 것을 봅니다.

"문서에서 수치를 뽑고, 3개가 맞는지 확인하고, 직원용 안내문으로 써 줘"를 한 번에 시킬 수도 있습니다. 그러면 셋 중 어디서 틀렸는지 알 수 없고, 모델이 개수를 잘못 세도 그냥 넘어갑니다.

아래 셀은 이 일을 셋으로 나눕니다.
- **① 뽑기** — v1으로 JSON을 뽑습니다 (모델)
- **② 검사** — 항목이 3개인지, 값이 숫자인지 셉니다 (**코드** — 세는 일은 모델이 자주 틀립니다)
- **③ 쓰기** — ②를 통과한 JSON만 보고 안내문을 씁니다 (모델). ②에서 걸리면 ③은 건너뛰고 ①로 돌아갑니다.

In [ ]:
# ① 추출 — 지금까지 쓴 v1을 그대로 쓴다
system, user = build(v1, doc)
r1 = llm.call(user, system=system)
try:
    data = parse(r1.text)
except ValueError as e:
    data = None
    print("① 추출 실패:", e)
print("① 추출:", json.dumps(data, ensure_ascii=False))

# ② 검증 — 모델이 아니라 코드가 센다
problems = []
if data is None:
    problems.append("JSON 아님")
else:
    if len(data.get("항목", [])) != 3:
        problems.append(f"항목 {len(data.get('항목', []))}개")
    for it in data.get("항목", []):
        if not isinstance(it.get("값"), (int, float)):
            problems.append(f"{it.get('이름')} 값이 숫자가 아님")
print("② 검증:", problems or "통과")

# ③ 작성 — 검증을 통과한 JSON만 안내문으로
if not problems:
    r3 = llm.call(
        "아래 JSON만 보고 직원용 메신저 안내문을 세 줄로 쓰세요. JSON에 없는 내용은 쓰지 않습니다.\n\n"
        + json.dumps(data, ensure_ascii=False),
        max_tokens=300)
    print("③ 작성:\n" + r3.text.strip())
else:
    print("③ 작성: 건너뜀 — ①로 돌아가 고칩니다")

## 7. 일반화된 프롬프트를 여러 다른 문서에 적용한다
v1은 문서 한 편에 맞춘 프롬프트가 아닙니다. 역할 · 할 일 · 출력 형식 · 예시는 고정하고 `{document}` 자리만 비워 둔 **일반화된 프롬프트**입니다.

아래 셀은 그 v1을 `data/docs/`의 문서 5개에 차례로 적용해, **프롬프트를 고치지 않고도 다른 문서에서 같은 모양의 결과가 나오는지** 확인합니다. 오후 블록 4는 같은 방법으로 문서 20개에 적용합니다. (이 셀은 아무것도 새로 저장하지 않습니다.)

In [ ]:
for p in sorted(Path("data/docs").glob("*.txt"))[:5]:
    system, user = build(v1, p.read_text(encoding="utf-8"))
    r = llm.call(user, system=system)
    try:
        out = parse(r.text)
        print(f"{p.name:<28} OK  ", [(i.get("이름"), i.get("값")) for i in out.get("항목", [])])
    except ValueError as e:
        print(f"{p.name:<28} 실패 {e}")

## 제출할 것
- `prompts/prompt_v0.txt` · `prompts/prompt_v1.txt` — 두 버전을 모두 남긴다
- v0에서 v1로 넘어오며 추가한 것과 그 이유 세 줄:
  1.
  2.
  3. 